In [64]:
import streamlit as st
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime , date
from dateutil.relativedelta import relativedelta
from datetime import timedelta
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
import itertools
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import tensorflow as tf 
from tf.keras.models import Sequential
from tf.keras.layers import LSTM, Dense

warnings.filterwarnings('ignore')
%matplotlib inline

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
df = yf.download('GOOGL' , start = '2025-04-18' , end='2026-04-17')
print(df.head())
print(df.columns)
print(df.index)
print(type(df.index))

In [ ]:
df.columns = df.columns.get_level_values(0) if isinstance(df.columns , pd.MultiIndex)else df.columns
df.index = pd.to_datetime(df.index)
df = df.sort_index()
df = df.dropna()
df

In [ ]:
from statsmodels.tsa.stattools import adfuller
ts = df['Close'].dropna()
res = adfuller(ts)
print('ADF Statistics' , res[0])    
print('p values' , res[1])
d = 0
while True:
    if(res[1]<=0.05):
        print(f'Stationary achieved' , res[1])
        break 
    d+=1
    ts = df['Close'].diff().dropna()
    res = adfuller(ts)
    print(f'--- Differencing Level: {d} ---')
    print(f'ADF Statistic: {res[0]}')
    print(f'p-value: {res[1]}')
    
# d = 1


In [ ]:
ts

In [ ]:
ts.values , ts.index 

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
plot_acf(ts)
plt.show()
plot_pacf(ts)
plt.show()

In [ ]:

train_size = int(len(ts) * 0.8)
train, test = ts[:train_size], ts[train_size:]
train.index,test.index

In [ ]:
best_rmse = None 
best_order = None
best_rmse = float("inf")
# split data
train_size = int(len(ts) * 0.8)
train, test = ts[:train_size], ts[train_size:]

for p in range(0,5):
    for q in range(0,5):
        model = ARIMA(train, order  = (p,1,q))
        res = model.fit()
        pred = res.forecast(steps=len(test))
        mse = mean_squared_error(test,pred)
        rmse = np.sqrt(mse)
        if(rmse<best_rmse):
            best_order = (p,1,q)
            best_rmse = rmse
            
print( 'Best order is : ', best_order)
print(rmse)

In [ ]:
# Rolling ARIMA..

history = list(train)
predictions = []
for i in range(len(test)):
    order = (2,1,0)
    final_model = ARIMA(history , order   = (2,1,0))
    res = final_model.fit()
    final_pred = float(res.forecast()[0]) # 1 step forecast
    predictions.append(final_pred) # append the predictions .
    history.append(float(test.iloc[i])) # append the history .
    
rmse = np.sqrt(mean_squared_error(test,predictions))
print(rmse)

## Base-Line Model

In [ ]:
order = (0,1,0)
baseline_model = ARIMA(train , order  = order)
res = baseline_model.fit()
baseline_pred = res.forecast(len(test))
mse = mean_squared_error(test,baseline_pred)
rmse = np.sqrt(mse)
print(rmse)

## Plot for ARIMA

In [ ]:
plt.plot(test.index,test, label = 'Original')
plt.plot(test.index,predictions, label = 'Predicted' , color ='red')
plt.plot(test.index,baseline_pred, label = 'Baseline' , color ='black')
plt.legend()
plt.title('Rolling ARIMA Forecast')
plt.show()

In [ ]:
test

## ARIMA PREDICTION for 30 Days

In [ ]:
series =  df['Close'].dropna()
# model_30 = ARIMA(series , order = (2,1,0))
# res = model_30.fit()
# forecasst_30 = res.forecast(steps = 30) 
history = list(df['Close'].dropna())
predictions_30 = []

order = (2,1,0)   # or your best_order

for i in range(30):
    model = ARIMA(history, order=order)
    res = model.fit()

    yhat = float(res.forecast()[0])
    predictions_30.append(yhat)

    history.append(yhat)   
future_index = pd.date_range(
    start=series.index[-1],
    periods=31,
    freq='B' 
)[1:]

plt.plot(series.index,series, label = 'Original' , color = 'red')
plt.plot(future_index, predictions_30,  label = 'Predicted 30 days ' , color = 'Black')
plt.legend()
plt.show()  

In [ ]:
len(predictions_30)

In [ ]:
len(df['Close']) , len(test) , len(train)

## LSTM


In [ ]:
df

In [ ]:
from sklearn.preprocessing import MinMaxScaler
def create_LSTM_sequences(df, window = 3):
    x = [] 
    y = []
    sequences = len(df) - window
    for i in range(sequences):
     x.append(df[i:i+window])
     y.append(df[i+window])     
    return np.array(x),np.array(y) 

def split(df,ratio=0.8):
    ts = pd.Series(df)
    train_size = int(len(ts) * ratio)
    train, test = ts[:train_size], ts[train_size:]
    return train,test 

def scaler(df,ratio=0.8):
    train,test = split(df,ratio)
    train =  train.values.reshape(-1,1)
    test = test.values.reshape(-1,1)
    scaler = MinMaxScaler(feature_range=(0,1))
    train_scaled = scaler.fit_transform(train)
    test_scaled = scaler.transform(test)
    return train_scaled , test_scaled ,scaler


def prepare_LSTM_model(X,y):
    # scale and create lstm sequences
    series = df['Close'].dropna()
    train_scaled, test_scaled , _ = scaler(series)
    data_scaled = np.vstack([train_scaled , test_scaled]) # merge the scaled data into one 
    X,y = create_LSTM_sequences(data_scaled , window=3)
    
    
    